# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [26]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania: 3.55s


In [28]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania (wątki): 0.77s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [20]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [21]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 0.35s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [1]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def get_cat_fact():
    return requests.get(CAT_API_URL).json().get("fact")


def run_sequential():
    print("Pobieranie sekwencyjne...")

    start = time.time()

    facts = []
    for _ in range(20):
        facts.append(get_cat_fact())

    end = time.time()

    print(f"Pobrano {len(facts)} faktów.")
    print(f"Czas sekwencyjny: {end - start:.2f}s")

    return facts, end - start


def run_threaded():
    print("\nPobieranie wielowątkowe...")

    start = time.time()

    with concurrent.futures.ThreadPoolExecutor() as executor:
        facts = list(executor.map(lambda _: get_cat_fact(), range(20)))

    end = time.time()

    print(f"Pobrano {len(facts)} faktów.")
    print(f"Czas wielowątkowy: {end - start:.2f}s")

    return facts, end - start


seq_facts, seq_time = run_sequential()
thread_facts, thread_time = run_threaded()

print("\nPorównanie:")
print(f"Sekwencyjnie: {seq_time:.2f}s")
print(f"Wielowątkowo: {thread_time:.2f}s")

if thread_time < seq_time:
    print(f"Wielowątkowo było szybciej o {seq_time - thread_time:.2f}s")
else:
    print(f"Sekwencyjnie było szybciej o {thread_time - seq_time:.2f}s")

print("\nPierwsze 5 faktów:")
for i, fact in enumerate(thread_facts[:5], 1):
    print(f"{i}. {fact}")

Pobieranie sekwencyjne...
Pobrano 20 faktów.
Czas sekwencyjny: 4.27s

Pobieranie wielowątkowe...
Pobrano 20 faktów.
Czas wielowątkowy: 2.17s

Porównanie:
Sekwencyjnie: 4.27s
Wielowątkowo: 2.17s
Wielowątkowo było szybciej o 2.10s

Pierwsze 5 faktów:
1. The silks created by weavers in Baghdad were inspired by the beautiful and varied colors and markings of cat coats. These fabrics were called 'tabby' by European traders.
2. A cat's appetite is the barometer of its health. Any cat that does not eat or drink for more than two days should be taken to a vet.
3. On average, a cat will sleep for 16 hours a day.
4. Unlike humans, cats are usually lefties. Studies indicate that their left paw is typically their dominant paw.
5. A cat can jump 5 times as high as it is tall.


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [3]:
import queue
import threading
import time

q = queue.Queue()
LICZBA_ELEMENTOW = 20

STOP = None

def producent():
    for i in range(LICZBA_ELEMENTOW):
        print(f"Producent -> {i}")
        q.put(i)
        time.sleep(0.1)

    q.put(STOP)
    q.put(STOP)

def konsument_parzyste():
    while True:
        liczba = q.get()

        if liczba is STOP:
            q.put(STOP)
            break

        if liczba % 2 == 0:
            print(f"Konsument parzyste <- {liczba}")
        else:
            q.put(liczba)

        q.task_done()

def konsument_nieparzyste():
    while True:
        liczba = q.get()

        if liczba is STOP:
            q.put(STOP)
            break

        if liczba % 2 != 0:
            print(f"Konsument nieparzyste <- {liczba}")
        else:
            q.put(liczba)

        q.task_done()

t_prod = threading.Thread(target=producent)
t_parz = threading.Thread(target=konsument_parzyste)
t_niep = threading.Thread(target=konsument_nieparzyste)

t_prod.start()
t_parz.start()
t_niep.start()

t_prod.join()
t_parz.join()
t_niep.join()

print("Koniec")

Producent -> 0
Konsument parzyste <- 0
Producent -> 1
Konsument nieparzyste <- 1
Producent -> 2
Konsument parzyste <- 2
Producent -> 3
Konsument nieparzyste <- 3
Producent -> 4
Konsument parzyste <- 4
Producent -> 5
Konsument nieparzyste <- 5
Producent -> 6
Konsument parzyste <- 6
Producent -> 7
Konsument nieparzyste <- 7
Producent -> 8
Konsument parzyste <- 8
Producent -> 9
Konsument nieparzyste <- 9
Producent -> 10
Konsument parzyste <- 10
Producent -> 11
Konsument nieparzyste <- 11
Producent -> 12
Konsument parzyste <- 12
Producent -> 13
Konsument nieparzyste <- 13
Producent -> 14
Konsument parzyste <- 14
Producent -> 15
Konsument nieparzyste <- 15
Producent -> 16
Konsument parzyste <- 16
Producent -> 17
Konsument nieparzyste <- 17
Producent -> 18
Konsument parzyste <- 18
Producent -> 19
Konsument nieparzyste <- 19
Koniec


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [5]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

numbers = range(1, 10001)

start = time.time()

with multiprocessing.Pool() as pool:
    results = pool.map(calculate_power_sum, numbers)

end = time.time()

print(f"Obliczono {len(results)} wyników.")
print(f"Czas wykonania: {end - start:.2f} s")

print("Pierwsze 5 wyników:")
for result in results[:5]:
    print(result)

Obliczono 10000 wyników.
Czas wykonania: 0.19 s
Pierwsze 5 wyników:
100
2535301200456458802993406410750
773066281098016996554691694648431909053161283000
2142584059011987034055949456454883470029603991710390447068500
9860761315262647567646607066034827870915080438862787559628486633300780
